## Gold Layer — Analyst-Ready Datamarts

This notebook builds 5 denormalized datamarts from the silver star schema. The gold layer is where **joins disappear** — each table is pre-aggregated or pre-joined so that dashboards, BI tools, and ad-hoc SQL can answer business questions with a single `SELECT ... WHERE`.

**What comes in:** 9 silver tables from `employeedatacatalog.silver_movie` (normalized star schema).

**What goes out:** 5 gold datamarts in `employeedatacatalog.gold_movie`.

---

### Datamarts
| Table | Rows | What It Answers |
| --- | --- | --- |
| `gold_movie_summary` | 9,770 | Everything about a movie in one row (financials + director + cast size + review stats) |
| `gold_genre_analytics` | 19 | Which genres perform best financially and critically? |
| `gold_director_analytics` | 1,563 | Which directors (2+ films) have the best track records? |
| `gold_yearly_trends` | 100 | How has the industry evolved year-over-year? |
| `gold_actor_analytics` | 10,535 | Which actors (3+ films) appear in the highest-performing movies? |

---

### Steps:
1. **Movie summary (wide table)** — Joins dim_movies + fact_movie_metrics + director (from crew) + cast count + review stats into one flat row per movie. This is the table most analyses start from.
2. **Genre analytics** — Aggregates via `bridge_movie_genres` so multi-genre films count toward each genre fairly.
3. **Director analytics** — Groups by director (2+ films), uses `row_number()` to attach their top-rated movie title.
4. **Yearly trends** — Simple GROUP BY year with avg/sum metrics. Useful for time-series visualizations.
5. **Actor analytics** — Groups by actor (3+ films), tracks top-billed count (cast_order = 0) as a star-power signal.
6. **Write to Delta** — All tables saved under `gold/movie/` and registered in Unity Catalog.

### Design Philosophy:
- **No joins required downstream** — analysts just query one table
- **Pre-computed metrics** — profit, ROI, multipliers already calculated
- **Threshold filters** — directors need 2+ films, actors need 3+ to avoid small-sample noise

In [0]:
from pyspark.sql.functions import *

# All three layers (bronze/silver/gold) share the same ADLS root.
# Gold is the final stop — pre-aggregated, denormalized, ready for dashboards.
root_path = "abfss://employee@dataanlysisazuredatalake.dfs.core.windows.net"
bronze_path = f"{root_path}/bronze"
silver_path = f"{root_path}/silver"
gold_path = f"{root_path}/gold"

bronze_sch = "bronze_movie"
silver_sch = "silver_movie"
gold_sch = "gold_movie"

movies_db = "employeedatacatalog"

# Shorthand so we don't repeat the full schema path in every query
S = f"{movies_db}.{silver_sch}"

In [0]:
# gold_movie_summary is the main table analysts will hit most often.
# It flattens the star schema into a single wide row per movie:
#   - Core attributes from dim_movies (title, release date, runtime, language)
#   - Financials from fact_movie_metrics (budget, revenue, profit, ROI)
#   - Director name from fact_movie_crew (first director if multiple)
#   - Cast size from fact_movie_cast
#   - Review stats from fact_movie_reviews (count + avg rating)
#
# This way a dashboard query is just SELECT ... WHERE ... — no joins needed.

gold_movie_summary = spark.sql(f"""
    WITH directors AS (
        -- A movie can have multiple directors (rare but happens).
        -- We grab the first one alphabetically to keep one row per movie.
        SELECT movie_id,
               first(p.person_name) AS director_name
        FROM   {S}.fact_movie_crew c
        JOIN   {S}.dim_people p ON c.person_id = p.person_id
        WHERE  c.job = 'Director'
        GROUP BY movie_id
    ),
    cast_counts AS (
        SELECT movie_id, count(*) AS cast_size
        FROM   {S}.fact_movie_cast
        GROUP BY movie_id
    ),
    review_stats AS (
        SELECT movie_id,
               count(*)              AS review_count,
               round(avg(author_rating), 2) AS avg_review_rating
        FROM   {S}.fact_movie_reviews
        GROUP BY movie_id
    )
    SELECT
        m.movie_id,
        m.title,
        m.original_title,
        m.release_date,
        year(m.release_date)        AS release_year,
        m.runtime_minutes,
        m.original_language,
        m.genre_list,
        m.status,
        m.is_adult,
        -- financials
        f.budget,
        f.revenue,
        f.profit,
        f.roi_pct,
        -- ratings
        f.vote_average,
        f.vote_count,
        f.popularity,
        -- enrichments from CTEs above
        d.director_name,
        cc.cast_size,
        r.review_count,
        r.avg_review_rating
    FROM      {S}.dim_movies m
    JOIN      {S}.fact_movie_metrics f  ON m.movie_id = f.movie_id
    LEFT JOIN directors d               ON m.movie_id = d.movie_id
    LEFT JOIN cast_counts cc            ON m.movie_id = cc.movie_id
    LEFT JOIN review_stats r            ON m.movie_id = r.movie_id
""")

print(f"gold_movie_summary: {gold_movie_summary.count():,} rows")
display(gold_movie_summary.filter(col("movie_id") == 28))  # sanity check: Apocalypse Now

In [0]:
# How does each genre stack up financially?
# Uses the bridge_movie_genres table so a movie tagged "Action,Comedy"
# contributes to BOTH genres. This gives a fair picture of which
# genres attract the biggest budgets and which actually pay off.

gold_genre_analytics = spark.sql(f"""
    SELECT
        bg.genre_name,
        count(DISTINCT bg.movie_id)          AS movie_count,
        round(avg(f.budget), 0)              AS avg_budget,
        round(avg(f.revenue), 0)             AS avg_revenue,
        round(avg(f.profit), 0)              AS avg_profit,
        sum(f.revenue)                       AS total_revenue,
        round(avg(f.vote_average), 2)        AS avg_rating,
        round(avg(f.popularity), 2)          AS avg_popularity
    FROM      {S}.bridge_movie_genres bg
    JOIN      {S}.fact_movie_metrics f ON bg.movie_id = f.movie_id
    GROUP BY  bg.genre_name
    ORDER BY  total_revenue DESC
""")

print(f"gold_genre_analytics: {gold_genre_analytics.count():,} rows")
display(gold_genre_analytics)

In [0]:
# For each director with 2+ films: how many movies, what's their
# average revenue, average rating, and what's their best-reviewed film?
# This is the table you'd use to answer "should we greenlight a project
# with director X?" — their track record is right here.
#
# max_by(title, vote_average) deterministically picks the title of the
# highest-rated movie — no need for row_number tricks.

gold_director_analytics = spark.sql(f"""
    WITH director_movies AS (
        SELECT
            c.person_id,
            p.person_name       AS director_name,
            m.title,
            f.revenue,
            f.budget,
            f.profit,
            f.vote_average,
            f.popularity
        FROM      {S}.fact_movie_crew c
        JOIN      {S}.dim_people p          ON c.person_id = p.person_id
        JOIN      {S}.dim_movies m          ON c.movie_id  = m.movie_id
        JOIN      {S}.fact_movie_metrics f  ON c.movie_id  = f.movie_id
        WHERE     c.job = 'Director'
    )
    SELECT
        person_id,
        director_name,
        count(*)                            AS movie_count,
        sum(revenue)                        AS total_revenue,
        round(avg(budget), 0)               AS avg_budget,
        round(avg(revenue), 0)              AS avg_revenue,
        round(avg(profit), 0)               AS avg_profit,
        round(avg(vote_average), 2)         AS avg_rating,
        round(avg(popularity), 2)           AS avg_popularity,
        -- best movie by rating (deterministic)
        max_by(title, vote_average)         AS top_movie
    FROM director_movies
    GROUP BY person_id, director_name
    HAVING count(*) >= 2
    ORDER BY avg_rating DESC
""")

print(f"gold_director_analytics: {gold_director_analytics.count():,} rows")
display(gold_director_analytics.limit(10))

In [0]:
# One row per release year with aggregate stats: how many movies came out,
# what was the average budget, revenue, profit, rating, and runtime.
# Perfect for time-series charts showing how the industry evolved —
# budget inflation, the 2020 pandemic crater, streaming era shifts, etc.

gold_yearly_trends = spark.sql(f"""
    SELECT
        year(m.release_date)                AS release_year,
        count(*)                            AS movie_count,
        round(avg(f.budget), 0)             AS avg_budget,
        round(avg(f.revenue), 0)            AS avg_revenue,
        round(avg(f.profit), 0)             AS avg_profit,
        sum(f.revenue)                      AS total_revenue,
        sum(f.budget)                       AS total_budget,
        round(avg(f.vote_average), 2)       AS avg_rating,
        round(avg(f.popularity), 2)         AS avg_popularity,
        round(avg(m.runtime_minutes), 0)    AS avg_runtime
    FROM      {S}.dim_movies m
    JOIN      {S}.fact_movie_metrics f ON m.movie_id = f.movie_id
    WHERE     m.release_date IS NOT NULL
    GROUP BY  year(m.release_date)
    ORDER BY  release_year
""")

print(f"gold_yearly_trends: {gold_yearly_trends.count():,} rows")
display(gold_yearly_trends.limit(10))

In [0]:
# Which actors show up the most and in the best-performing films?
# top_billed_count tracks how often they were the lead (cast_order = 0),
# which is a better signal of star power than just appearing in the credits.
# Filtered to actors with 3+ movies so we're not fooled by small samples.

gold_actor_analytics = spark.sql(f"""
    SELECT
        ca.person_id,
        p.person_name                       AS actor_name,
        count(DISTINCT ca.movie_id)         AS movie_count,
        sum(CASE WHEN ca.cast_order = 0 THEN 1 ELSE 0 END) AS top_billed_count,
        round(avg(f.vote_average), 2)       AS avg_movie_rating,
        round(avg(f.popularity), 2)         AS avg_movie_popularity,
        sum(f.revenue)                      AS total_revenue,
        round(avg(f.revenue), 0)            AS avg_revenue
    FROM      {S}.fact_movie_cast ca
    JOIN      {S}.dim_people p          ON ca.person_id = p.person_id
    JOIN      {S}.fact_movie_metrics f  ON ca.movie_id  = f.movie_id
    GROUP BY  ca.person_id, p.person_name
    HAVING    count(DISTINCT ca.movie_id) >= 3
    ORDER BY  avg_movie_rating DESC
""")

print(f"gold_actor_analytics: {gold_actor_analytics.count():,} rows")
display(gold_actor_analytics.limit(10))

In [0]:
# Write all five gold datamarts to Delta Lake and register them in Unity Catalog.
# These are the tables that dashboards, BI tools, and analysts should query —
# no need to touch silver or bronze directly once these exist.
#
# Stored under gold/movie/ to avoid path collisions with other projects.

gold_tables = {
    "gold_movie_summary":      gold_movie_summary,
    "gold_genre_analytics":    gold_genre_analytics,
    "gold_director_analytics": gold_director_analytics,
    "gold_yearly_trends":      gold_yearly_trends,
    "gold_actor_analytics":    gold_actor_analytics,
}

for name, df in gold_tables.items():
    table_path = f"{gold_path}/movie/{name}"
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(table_path)
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {movies_db}.{gold_sch}.{name}
                  USING DELTA LOCATION '{table_path}'""")
    print(f"Saved {movies_db}.{gold_sch}.{name}")

print("\nGold layer complete!")